# Logged-data observational experiment with doubly robust policy evaluation

This notebook corrects the previous observational experiment. It trains joint and decoupled policies from **one logged outcome per training customer**, then evaluates those learned policies on a separate logged evaluation set.

The doubly robust (DR) estimator is used for **off-policy evaluation**, its standard role: it estimates the incremental value of a fixed learned policy from logged outcomes. It is not used as a high-variance pseudo-outcome policy learner.

The behavior policy mixes a decoupled production-style policy with random joint exploration. The experiment tests how overlap affects the accuracy of direct-method and DR policy-value estimates, while hidden potential outcomes provide ground truth only for simulation evaluation.


In [1]:
import sys
import subprocess

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost>=2.0.0", "--quiet"])
    import xgboost as xgb

import numpy as np
import pandas as pd

from scipy.special import expit, softmax
from sklearn.dummy import DummyRegressor
from xgboost import XGBRegressor

pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
print(f"XGBoost version: {xgb.__version__}")


XGBoost version: 3.2.0


In [2]:
# Vertex GPU preflight: this notebook requires CUDA.
try:
    _gpu_check = XGBRegressor(
        objective="reg:squarederror", n_estimators=1,
        tree_method="hist", device="cuda", random_state=0,
    )
    _gpu_check.fit(np.array([[0.0], [1.0]]), np.array([0.0, 1.0]))
    del _gpu_check
    print("CUDA XGBoost check passed.")
except Exception as exc:
    raise RuntimeError(
        "CUDA is unavailable to XGBoost. In Vertex AI, select a GPU accelerator "
        "and restart the kernel before running this notebook."
    ) from exc


/opt/conda/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [21:14:56] WARNING: /__w/xgboost/xgboost/src/context.cc:53: No visible GPU is found, setting device to CPU.
  bst.update(dtrain, iteration=i, fobj=obj)
/opt/conda/lib/python3.10/site-packages/xgboost/training.py:200: UserWarning: [21:14:56] WARNING: /__w/xgboost/xgboost/src/context.cc:207: Device is changed from GPU to CPU as we couldn't find any available GPU on the system.
  bst.update(dtrain, iteration=i, fobj=obj)


CUDA XGBoost check passed.


In [3]:
# Final experiment configuration
N = 10_000
D = 10
R = 10
O = 3
N_SEEDS = 20

LAMBDA_VAL = 0.75
BETA_VAL = 0.50
DISCOUNT = np.array([0.00, 0.10, 0.25])
SIGMA_FRAC = 0.15
BASELINE_ACTION = 0  # action (r=0, o=0)
MIN_ACTION_SAMPLES = 20

# Larger epsilon means more randomized exploration and stronger overlap.
EXPLORATION_GRID = [0.50, 0.25, 0.10]

rng = np.random.RandomState(0)
W_AFFINITY = rng.randn(R, D) * 0.5
V_PRICE = rng.randn(D) * 0.5
W_BASELINE = rng.randn(R, D) * 0.3
B_BASELINE = rng.uniform(20, 80, size=R)

rng_w = np.random.RandomState(99)
W_COUPLING = rng_w.randn(R, O)
W_COUPLING -= W_COUPLING.mean(axis=1, keepdims=True)
assert np.allclose(W_COUPLING.sum(axis=1), 0.0, atol=1e-10)


In [4]:
# Population and hidden response surface
def make_population(n=N):
    rng_x = np.random.RandomState(42)
    a = rng_x.randn(D, D)
    sigma_x = a @ a.T / D + np.eye(D)
    x = rng_x.multivariate_normal(np.zeros(D), sigma_x, size=n)
    x = (x - x.mean(axis=0)) / x.std(axis=0)
    affinity = softmax(x @ W_AFFINITY.T, axis=1)
    price_sensitivity = expit(x @ V_PRICE)
    baseline = np.abs(x @ W_BASELINE.T + B_BASELINE) + 10
    return x, affinity, price_sensitivity, baseline


def simulate_potential_outcomes(affinity, price_sensitivity, baseline, seed):
    """Hidden Y_i(r,o), used only to construct logged samples and simulation truth."""
    rng_y = np.random.RandomState(seed)
    n = len(affinity)
    y = np.zeros((n, R, O))
    sigma_y = SIGMA_FRAC * baseline.mean()
    mean_baseline = baseline.mean(axis=1)
    for r in range(R):
        for o in range(O):
            main = baseline[:, r] + price_sensitivity * DISCOUNT[o] * mean_baseline
            interaction = affinity[:, r] * baseline[:, r] * W_COUPLING[r, o] * price_sensitivity
            y[:, r, o] = np.maximum(
                main + LAMBDA_VAL * interaction + rng_y.normal(0, sigma_y, n), 0.0
            )
    return y


In [5]:
# Behavior policy and logged-data construction
def log_actions(affinity, price_sensitivity, exploration, seed):
    """Decoupled behavior policy plus epsilon random joint exploration.

    e(a|x) = (1-epsilon) p_rec(r|x)p_offer(o|x) + epsilon/(R*O).
    The exploration term ensures a known positive lower bound for every joint action.
    """
    rng_log = np.random.RandomState(seed)
    n = len(affinity)
    p_rec = affinity
    raw_offer = np.column_stack([1.0 - price_sensitivity, 0.60 * price_sensitivity, 0.40 * price_sensitivity])
    p_offer = raw_offer / raw_offer.sum(axis=1, keepdims=True)
    p_decoupled = (p_rec[:, :, None] * p_offer[:, None, :]).reshape(n, R * O)
    p_joint = (1.0 - exploration) * p_decoupled + exploration / (R * O)
    action = np.array([rng_log.choice(R * O, p=p_joint[i]) for i in range(n)])
    return action, p_joint


def observed_outcome(y, action):
    return y.reshape(len(y), R * O)[np.arange(len(y)), action]


def make_gbr(seed):
    return XGBRegressor(
        objective="reg:squarederror",
        n_estimators=100,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=10,
        subsample=0.90,
        colsample_bytree=0.90,
        tree_method="hist",
        device="cuda",
        random_state=seed,
        n_jobs=1,
        verbosity=0,
    )


In [6]:
# Learners trained on the logged training sample only
def fit_joint_outcome_models(x, y_obs, action, seed):
    models = []
    for a in range(R * O):
        rows = np.flatnonzero(action == a)
        if len(rows) == 0:
            models.append(None)
        else:
            model = DummyRegressor(strategy="mean") if len(rows) < MIN_ACTION_SAMPLES else make_gbr(seed + a)
            model.fit(x[rows], y_obs[rows])
            models.append(model)
    if models[BASELINE_ACTION] is None:
        raise RuntimeError("The baseline action was never logged.")
    return models


def predict_joint(models, x):
    out = np.full((len(x), R * O), np.nan)
    for a, model in enumerate(models):
        if model is not None:
            out[:, a] = model.predict(x)
    return out


def fit_decoupled_models(x, y_obs, action, seed):
    r_log, o_log = action // O, action % O
    rec_models, offer_models = [], []
    for r in range(R):
        rows = np.flatnonzero(r_log == r)
        model = DummyRegressor(strategy="mean") if len(rows) < MIN_ACTION_SAMPLES else make_gbr(seed + r)
        model.fit(x[rows], y_obs[rows])
        rec_models.append(model)
    for o in range(O):
        rows = np.flatnonzero(o_log == o)
        model = DummyRegressor(strategy="mean") if len(rows) < MIN_ACTION_SAMPLES else make_gbr(seed + 100 + o)
        model.fit(x[rows], y_obs[rows])
        offer_models.append(model)
    baseline_rows = np.flatnonzero(action == BASELINE_ACTION)
    baseline = DummyRegressor(strategy="mean") if len(baseline_rows) < MIN_ACTION_SAMPLES else make_gbr(seed + 200)
    baseline.fit(x[baseline_rows], y_obs[baseline_rows])
    return rec_models, offer_models, baseline


In [7]:
# Policy construction and OPE estimators
def top_budget(scores, beta):
    n = len(scores)
    target = scores.argmax(axis=1)
    treated = np.zeros(n, dtype=bool)
    treated[np.argsort(scores.max(axis=1))[::-1][:max(1, int(beta * n))]] = True
    target[~treated] = BASELINE_ACTION
    return target


def joint_policy(models, x):
    mu = predict_joint(models, x)
    scores = mu - mu[:, [BASELINE_ACTION]]
    scores[:, BASELINE_ACTION] = 0.0
    return top_budget(np.nan_to_num(scores, nan=-np.inf), BETA_VAL), mu


def decoupled_policy(models, x):
    rec_models, offer_models, baseline = models
    baseline_hat = baseline.predict(x)
    rec = np.column_stack([m.predict(x) for m in rec_models]) - baseline_hat[:, None]
    offer = np.column_stack([m.predict(x) for m in offer_models]) - baseline_hat[:, None]
    r_star, o_star = rec.argmax(axis=1), offer.argmax(axis=1)
    pair = r_star * O + o_star
    rank = rec.max(axis=1) + offer.max(axis=1)
    treated = np.zeros(len(x), dtype=bool)
    treated[np.argsort(rank)[::-1][:max(1, int(BETA_VAL * len(x)))]] = True
    pair[~treated] = BASELINE_ACTION
    return pair, baseline_hat


def true_incremental_value(y_hidden, action):
    flat = y_hidden.reshape(len(y_hidden), R * O)
    return (flat[np.arange(len(action)), action] - flat[:, BASELINE_ACTION]).mean()


def ope_values(models, x_eval, y_obs_eval, action_eval, propensity_eval, target_action):
    """Direct-method and DR estimates of policy value minus baseline-policy value."""
    mu = predict_joint(models, x_eval)
    row = np.arange(len(x_eval))
    observed_mu = mu[row, action_eval]
    if np.isnan(observed_mu).any():
        raise RuntimeError("Evaluation set contains an action absent from logged training data.")
    base_mu = mu[:, BASELINE_ACTION]
    target_mu = mu[row, target_action]
    dm = (target_mu - base_mu).mean()

    e_observed = propensity_eval[row, action_eval]
    residual = y_obs_eval - observed_mu
    target_match = (action_eval == target_action)
    base_match = (action_eval == BASELINE_ACTION)
    dr_target = target_mu + target_match * residual / e_observed
    dr_base = base_mu + base_match * residual / e_observed
    return dm, (dr_target - dr_base).mean()


In [8]:
# Final simulation: 20 independent seeds × 3 exploration conditions
x, affinity, price_sensitivity, baseline = make_population(N)
all_rows = []

for seed in range(N_SEEDS):
    split_rng = np.random.RandomState(1000 + seed)
    index = split_rng.permutation(N)
    train_idx, eval_idx = index[:int(0.60 * N)], index[int(0.60 * N):]
    x_train, x_eval = x[train_idx], x[eval_idx]
    aff_train, aff_eval = affinity[train_idx], affinity[eval_idx]
    ps_train, ps_eval = price_sensitivity[train_idx], price_sensitivity[eval_idx]
    base_train, base_eval = baseline[train_idx], baseline[eval_idx]
    y_train = simulate_potential_outcomes(aff_train, ps_train, base_train, 100 * seed)
    y_eval = simulate_potential_outcomes(aff_eval, ps_eval, base_eval, 100 * seed + 1)

    for epsilon in EXPLORATION_GRID:
        a_train, p_train = log_actions(aff_train, ps_train, epsilon, 10_000 * seed + int(epsilon * 1000))
        a_eval, p_eval = log_actions(aff_eval, ps_eval, epsilon, 20_000 * seed + int(epsilon * 1000))
        y_train_obs = observed_outcome(y_train, a_train)
        y_eval_obs = observed_outcome(y_eval, a_eval)

        joint_models = fit_joint_outcome_models(x_train, y_train_obs, a_train, 30_000 * seed)
        dec_models = fit_decoupled_models(x_train, y_train_obs, a_train, 40_000 * seed)
        joint_action, _ = joint_policy(joint_models, x_eval)
        dec_action, _ = decoupled_policy(dec_models, x_eval)

        true_joint = true_incremental_value(y_eval, joint_action)
        true_dec = true_incremental_value(y_eval, dec_action)
        dm_joint, dr_joint = ope_values(joint_models, x_eval, y_eval_obs, a_eval, p_eval, joint_action)
        dm_dec, dr_dec = ope_values(joint_models, x_eval, y_eval_obs, a_eval, p_eval, dec_action)

        all_rows.append({
            "seed": seed, "exploration": epsilon,
            "min_logged_propensity": p_eval.min(),
            "pct_logged_prop_lt_0p01": 100 * np.mean(p_eval[np.arange(len(a_eval)), a_eval] < 0.01),
            "true_joint_irc": true_joint, "true_decoupled_irc": true_dec,
            "true_joint_advantage": true_joint - true_dec,
            "dm_joint_ope": dm_joint, "dr_joint_ope": dr_joint,
            "dm_decoupled_ope": dm_dec, "dr_decoupled_ope": dr_dec,
            "abs_error_dm_joint": abs(dm_joint - true_joint),
            "abs_error_dr_joint": abs(dr_joint - true_joint),
        })

    pd.DataFrame(all_rows).to_csv("results_observational_ope_partial.csv", index=False)
    print(f"Completed seed {seed + 1}/{N_SEEDS}")

df_ope = pd.DataFrame(all_rows)
df_ope.to_csv("results_observational_ope.csv", index=False)
print("Saved results_observational_ope.csv")


Completed seed 1/20
Completed seed 2/20
Completed seed 3/20
Completed seed 4/20
Completed seed 5/20
Completed seed 6/20
Completed seed 7/20
Completed seed 8/20
Completed seed 9/20
Completed seed 10/20
Completed seed 11/20
Completed seed 12/20
Completed seed 13/20
Completed seed 14/20
Completed seed 15/20
Completed seed 16/20
Completed seed 17/20
Completed seed 18/20
Completed seed 19/20
Completed seed 20/20
Saved results_observational_ope.csv


In [9]:
# Paper-ready summary
summary = df_ope.groupby("exploration", as_index=False).agg(
    true_joint_advantage_mean=("true_joint_advantage", "mean"),
    true_joint_advantage_sd=("true_joint_advantage", "std"),
    abs_error_dm_joint_mean=("abs_error_dm_joint", "mean"),
    abs_error_dr_joint_mean=("abs_error_dr_joint", "mean"),
    pct_logged_prop_lt_0p01_mean=("pct_logged_prop_lt_0p01", "mean"),
)
display(summary.round(4))

print("Interpretation:")
print("• The joint-vs-decoupled advantage is assessed against hidden simulation truth.")
print("• DM and DR are off-policy value estimators, not competing policy learners.")
print("• DR should be interpreted alongside overlap diagnostics; it need not dominate under every finite-sample condition.")


,exploration,true_joint_advantage_mean,true_joint_advantage_sd,abs_error_dm_joint_mean,abs_error_dr_joint_mean,pct_logged_prop_lt_0p01_mean
0,0.1000,1.0907,0.3154,4.3982,1.4200,7.6275
1,0.2500,1.0719,0.3363,3.8982,1.1243,7.4362
2,0.5000,0.9154,0.2977,3.3027,1.1851,0.0000


Interpretation:
• The joint-vs-decoupled advantage is assessed against hidden simulation truth.
• DM and DR are off-policy value estimators, not competing policy learners.
• DR should be interpreted alongside overlap diagnostics; it need not dominate under every finite-sample condition.
